# SOC Trajectory Visualization: Panasonic25 -> CMU eVTOL Transfer

Per-mission SOC (state of charge) trajectory plots for the seven CMU
"core seven" target cells, comparing the ground-truth reference SOC against
six transfer-learning frameworks: Vidal, DTNN, TATN, SFTNN, DARM, and
Backbone.

**Each subplot shows exactly one representative test mission for one
battery cell** -- not the whole test-set lifespan. An earlier version of
this notebook concatenated every test mission chronologically into a single
subplot; with 100-450+ missions per cell the result was an unreadable wall
of overlapping color. This version fixes that by selecting one mission per
cell (see the mission-selection cell below) and plotting only that
mission's SOC trajectory against real elapsed time.

**Scope of this notebook (deliberately excluded):** RMSE/MAE charts,
framework ranking, statistical testing, and training/validation loss curves.
Those live in each framework's own benchmark notebook
(`32_tatn_panasonic25_to_cmu.ipynb`, `33_bian_dtnn_panasonic25_to_cmu.ipynb`,
`34_backbone_transfer_panasonic_to_cmu.ipynb`,
`35_sftnn_med_transfer_panasonic_to_cmu.ipynb`,
`36_liu_darm_transfer_panasonic_to_cmu.ipynb`). This notebook only draws
SOC-vs-time trajectories.

**Data availability as of 2026-09-18 (checked against Google Drive
`battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/`):**

| Framework | Status | Drive folder |
|---|---|---|
| DARM | available | `liu_2025_darm_ns_relabel_v2_e30_clean/test_predictions.parquet` |
| SFTNN | available | `sftnn_med_transfer_ns_relabel_v2/test_predictions.parquet` |
| Backbone | available | `backbone_transfer_ns_relabel_v2/test_predictions.parquet` |
| TATN | **not yet run** | `tatn_ns_relabel_v1/` does not exist yet on Drive (blocked on locating the official `pre-25.pt` checkpoint) |
| DTNN (Bian et al. 2021) | **not yet run** | `bian_2021_cmu/` (the script's own default output-root name) does not exist yet on Drive |
| Vidal | **not yet run** | no `panasonic_to_cmu_evtol`-scoped Vidal output exists yet |

The loading code does **not** hard-fail on a missing framework: it prints a
warning and simply omits that framework's line from the plots.

In [ ]:
"""Code Cell 1 -- imports and plotting style."""

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec

# Running on Colab: mount Google Drive so DATA_ROOT below resolves.
# No-op (and harmless) outside Colab.
try:
    import google.colab  # noqa: F401

    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)

pd.set_option("display.max_columns", 50)


In [ ]:
"""Code Cell 2 -- centralized configuration.

Every path, column name, filter value, and display choice used below lives
here so the rest of the notebook never hard-codes a string. Update this
cell (not the functions further down) when a new framework's results land
on Drive.
"""

# --- Where the benchmark outputs live -------------------------------------
# Confirmed against Google Drive on 2026-09-18: My Drive ->
# battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol/<run_root>/
# On Colab this is the path after drive.mount("/content/drive").
# TODO: update this path if the user's Drive layout changes.
DATA_ROOT = Path(
    "/content/drive/MyDrive/battery_soc_benchmark/outputs/benchmarks/panasonic_to_cmu_evtol"
)

# One top-level, already-aggregated test_predictions.parquet per framework.
# `None` means "not run yet" -- load_all_predictions() skips it with a
# warning rather than failing. Fill these in as new runs land on Drive.
RESULT_FILES: dict[str, Path | None] = {
    "Vidal": None,  # TODO: update this path once Vidal has been run against the CMU eVTOL target set
    "DTNN": None,  # TODO: update this path once run_bian_cmu.py's default output root ("bian_2021_cmu") exists on Drive
    "TATN": None,  # TODO: update this path once tatn_ns_relabel_v1/ exists on Drive (blocked on the official pre-25.pt checkpoint)
    "SFTNN": DATA_ROOT / "sftnn_med_transfer_ns_relabel_v2" / "test_predictions.parquet",
    "DARM": DATA_ROOT / "liu_2025_darm_ns_relabel_v2_e30_clean" / "test_predictions.parquet",
    "Backbone": DATA_ROOT / "backbone_transfer_ns_relabel_v2" / "test_predictions.parquet",
}

# The CMU "core seven" target cells (same order/labels as
# battery_benchmark.benchmarks.cmu_diagnostics.CELL_CONDITIONS).
CELL_IDS = ["VAH01", "VAH17", "VAH27", "VAH30", "VAH15", "VAH12", "VAH11"]

# Display order for the legend and for consistent per-framework colors.
FRAMEWORK_ORDER = ["Vidal", "DTNN", "TATN", "SFTNN", "DARM", "Backbone"]

# --- Column names in each framework's test_predictions.parquet -----------
COL_CELL_ID = "cell_id"
COL_METHOD = "method"
COL_MODEL = "model"
COL_BUDGET = "budget"
COL_SEED = "seed"
COL_SOC_REFERENCE = "soc_reference"
COL_SOC_PREDICTION = "soc_prediction"
COL_MISSION_ID = "mission_cycle_id"
COL_MISSION_SEQUENCE_ORDER = "mission_sequence_order"
COL_TIMESTEP_INDEX = "timestep_index"
COL_SOURCE_ROW_RIGHT = "source_row_right"
COL_ELAPSED_TIME_CANDIDATES = [
    "elapsed_candidate_s",
    "elapsed_candidate_s_end",
    "original_time_s",
    "timestamp",
]

# --- Exact per-framework filters (confirmed against the real schema) -----
# Every framework here is matched on an *exact* value, never a guessed
# candidate list. A framework not listed (Vidal / DTNN / TATN -- no data
# yet) falls back to a permissive "first method value present" match so
# the notebook doesn't hard-fail once their data lands; update this dict
# with the confirmed exact value(s) at that point instead of relying on
# the fallback.
FRAMEWORK_FILTER_SPECS: dict[str, dict[str, object]] = {
    "SFTNN": {COL_METHOD: "med"},  # never "zero_shot" / "dann_mmd"
    "DARM": {COL_METHOD: "DARM"},  # never "Source only"
    "Backbone": {
        COL_MODEL: "transformer_encoder",  # see BACKBONE_MODEL below
        COL_METHOD: "fine_tuned",
        COL_BUDGET: "full",
    },
}
BACKBONE_MODEL = "transformer_encoder"
BACKBONE_METHOD = "fine_tuned"
BACKBONE_BUDGET = "full"
FRAMEWORK_FILTER_SPECS["Backbone"] = {
    COL_MODEL: BACKBONE_MODEL,
    COL_METHOD: BACKBONE_METHOD,
    COL_BUDGET: BACKBONE_BUDGET,
}

# All frameworks are shown for the same seed.
PLOT_SEED = 42

# Each framework's own primary-evaluation output convention (used only to
# pick which framework's rows serve as the Reference SOC source -- never
# used to interpolate or resample any framework's predictions).
EVALUATION_UNIT: dict[str, str] = {
    "SFTNN": "all_timesteps",
    "DARM": "last_step_per_window",
    "Backbone": "all_timesteps",
    # Vidal / DTNN / TATN: unknown until each has a confirmed CMU-transfer run.
}

# --- Representative-mission selection --------------------------------------
# "midpoint": deterministic, no manual cherry-picking -- among the test
# missions common to every currently-loaded framework for a cell, sorted by
# mission_sequence_order, pick the one at position (N - 1) // 2.
MISSION_SELECTION_MODE = "midpoint"

# Force a specific mission_sequence_order for a specific cell, bypassing
# the midpoint rule. Empty by default. A mission listed here that no
# framework has at all raises ValueError (no silent fallback); one that
# exists for only some frameworks prints a warning and simply omits the
# others for that cell.
MISSION_OVERRIDE: dict[str, int] = {
    # "VAH01": 123,
    # "VAH17": 456,
}

# --- Reference-SOC cross-check -------------------------------------------
REFERENCE_SOC_MISMATCH_TOLERANCE = 1e-3

# --- Display-only downsampling (never alters underlying data) ------------
# Each subplot now draws a single mission, so the default is "show every
# point". Raise this only if a single mission still has too many points to
# render cleanly; first and last points are always kept regardless.
PLOT_EVERY_N = 1

# --- Output ----------------------------------------------------------------
OUTPUT_DIR = Path("./soc_trajectory_outputs")
FIG_DPI = 300
SAVE_INDIVIDUAL_CELL_FIGURES = False

# --- Colors ------------------------------------------------------------
REFERENCE_COLOR = "#1a1a1a"
FRAMEWORK_COLORS: dict[str, str] = {
    "Vidal": "#1f77b4",
    "DTNN": "#ff7f0e",
    "TATN": "#2ca02c",
    "SFTNN": "#d62728",
    "DARM": "#9467bd",
    "Backbone": "#8c564b",
}


In [ ]:
"""Code Cell 3 -- robust multi-format loading.

Reading every column of a multi-hundred-MB parquet is wasteful when we only
need a handful of fields, so `_read_predictions_file` projects columns up
front via pyarrow (through pandas' `columns=` kwarg). A framework whose file
is missing, unreadable, missing a required column, or with zero rows left
after filtering is skipped with `warnings.warn(...)`, never a hard crash.

Filtering here is exact-match against FRAMEWORK_FILTER_SPECS (SFTNN/DARM/
Backbone) plus PLOT_SEED -- no framework's rows are interpolated or
resampled to match another framework's point count; each framework's own
real prediction points are kept as-is.
"""

_REQUIRED_COLUMNS = {
    COL_CELL_ID,
    COL_METHOD,
    COL_SOC_REFERENCE,
    COL_SOC_PREDICTION,
    COL_MISSION_ID,
    COL_MISSION_SEQUENCE_ORDER,
    COL_TIMESTEP_INDEX,
}
_OPTIONAL_COLUMNS = {COL_MODEL, COL_BUDGET, COL_SEED, COL_SOURCE_ROW_RIGHT, *COL_ELAPSED_TIME_CANDIDATES}


def _parquet_columns(path: Path) -> set[str]:
    """Schema-only read (no row data) so we know which optional columns
    this particular file actually has, without guessing."""
    import pyarrow.parquet as pq

    return set(pq.ParquetFile(str(path)).schema.names)


def _ensure_percent_scale(series: pd.Series) -> pd.Series:
    """SOC is stored as a 0-1 fraction in the shared parquet schema
    (`metric_units` in run_metadata.json says "fraction"); auto-convert to
    percent only if the data actually looks fraction-scaled, so this stays
    safe even if a future run already stores percent."""
    finite = series.dropna()
    if finite.empty:
        return series * 100.0
    if finite.max() <= 1.5:
        return series * 100.0
    return series


def _read_predictions_file(framework: str, path: Path | None) -> pd.DataFrame | None:
    """Read one framework's test_predictions.parquet, already narrowed to
    this framework's exact method/model/budget filter (or, for a framework
    with no confirmed filter spec yet, the first `method` value present)
    and to PLOT_SEED. Returns None (with a warning) on any problem."""
    if path is None:
        warnings.warn(f"[{framework}] no result path configured yet -- skipping.")
        return None
    if not Path(path).exists():
        warnings.warn(f"[{framework}] file not found at {path} -- skipping.")
        return None

    try:
        available_columns = _parquet_columns(path)
    except Exception as exc:  # noqa: BLE001 -- intentionally broad: never crash the notebook
        warnings.warn(f"[{framework}] could not read schema of {path}: {exc!r} -- skipping.")
        return None

    missing_required = _REQUIRED_COLUMNS - available_columns
    if missing_required:
        warnings.warn(
            f"[{framework}] {path} is missing required columns {missing_required} -- skipping."
        )
        return None

    columns = sorted(_REQUIRED_COLUMNS | (_OPTIONAL_COLUMNS & available_columns))
    try:
        frame = pd.read_parquet(path, columns=columns)
    except Exception as exc:  # noqa: BLE001
        warnings.warn(f"[{framework}] failed reading {path}: {exc!r} -- skipping.")
        return None

    filter_spec = FRAMEWORK_FILTER_SPECS.get(framework)
    if filter_spec is not None:
        missing_filter_columns = set(filter_spec) - set(frame.columns)
        if missing_filter_columns:
            warnings.warn(
                f"[{framework}] cannot apply filter {filter_spec} -- missing columns "
                f"{missing_filter_columns} -- skipping."
            )
            return None
        mask = pd.Series(True, index=frame.index)
        for column, value in filter_spec.items():
            mask &= frame[column] == value
        frame = frame[mask]
        if frame.empty:
            warnings.warn(f"[{framework}] no rows matched filter {filter_spec} -- skipping.")
            return None
    else:
        # No confirmed filter spec yet (Vidal / DTNN / TATN): fall back to
        # "first method value present", purely so the notebook stays usable
        # the moment a first run lands -- replace with an exact
        # FRAMEWORK_FILTER_SPECS entry once the real value is known.
        available_methods = frame[COL_METHOD].dropna().unique().tolist()
        if not available_methods:
            warnings.warn(f"[{framework}] {path} has no non-null `method` values -- skipping.")
            return None
        chosen_method = available_methods[0]
        warnings.warn(
            f"[{framework}] no FRAMEWORK_FILTER_SPECS entry yet -- defaulting to the first "
            f"`method` value found ({chosen_method!r}) out of {available_methods}. "
            f"Add an exact filter to FRAMEWORK_FILTER_SPECS once confirmed."
        )
        frame = frame[frame[COL_METHOD] == chosen_method]

    if COL_SEED in frame.columns:
        frame = frame[frame[COL_SEED] == PLOT_SEED]
        if frame.empty:
            warnings.warn(f"[{framework}] no rows left with seed == {PLOT_SEED} -- skipping.")
            return None

    frame = frame[frame[COL_CELL_ID].isin(CELL_IDS)]
    if frame.empty:
        warnings.warn(f"[{framework}] no rows left after restricting to CELL_IDS -- skipping.")
        return None

    frame = frame.copy()
    frame[COL_SOC_REFERENCE] = _ensure_percent_scale(frame[COL_SOC_REFERENCE])
    frame[COL_SOC_PREDICTION] = _ensure_percent_scale(frame[COL_SOC_PREDICTION])
    frame["framework"] = framework
    return frame


def load_all_predictions() -> dict[str, pd.DataFrame]:
    """One DataFrame per framework that has usable data, keyed by framework
    name. Frameworks with no data yet are simply absent from the result --
    every downstream function must tolerate a missing key."""
    result: dict[str, pd.DataFrame] = {}
    for framework in FRAMEWORK_ORDER:
        frame = _read_predictions_file(framework, RESULT_FILES.get(framework))
        if frame is not None:
            result[framework] = frame
    if not result:
        warnings.warn(
            "No framework results were loadable yet -- check RESULT_FILES in "
            "the config cell. Plot cells below will render an empty legend "
            "until at least one framework's data is available."
        )
    return result


all_predictions = load_all_predictions()
print("Loaded frameworks:", list(all_predictions.keys()))
for _framework, _frame in all_predictions.items():
    print(f"  {_framework}: {len(_frame):,} rows across {_frame[COL_CELL_ID].nunique()} cells")


In [ ]:
"""Code Cell 4 -- representative-mission selection.

Each subplot now shows exactly one mission per cell, chosen deterministically
so nobody has to cherry-pick "the mission that looks best":

1. For a given cell, take the test missions (by mission_sequence_order)
   that are COMMON to every currently-loaded framework -- never let
   different frameworks draw different missions in the same subplot.
2. Sort those common missions ascending by mission_sequence_order.
3. MISSION_SELECTION_MODE == "midpoint" (the default): pick the mission at
   position (N - 1) // 2 -- the chronological midpoint of the *common* set.
4. MISSION_OVERRIDE[cell_id], if set, takes priority over the midpoint
   rule. A forced mission that no framework has at all raises ValueError
   (never a silent fallback); one that only some frameworks have prints a
   warning and simply omits the others for that cell.

No `plot_x` / `mission_slot` / chronological concatenation / mission-
boundary logic remains -- a subplot draws one mission's real elapsed time,
nothing more.
"""


def _cell_mission_sets(cell_id: str) -> dict[str, set[int]]:
    result = {}
    for framework, frame in all_predictions.items():
        cell_frame = frame[frame[COL_CELL_ID] == cell_id]
        if cell_frame.empty:
            continue
        result[framework] = set(cell_frame[COL_MISSION_SEQUENCE_ORDER].dropna().astype(int).unique())
    return result


def _find_common_missions(cell_id: str) -> list[int]:
    mission_sets = list(_cell_mission_sets(cell_id).values())
    if not mission_sets:
        return []
    common = mission_sets[0]
    for other in mission_sets[1:]:
        common = common & other
    return sorted(common)


def select_representative_mission(cell_id: str) -> dict:
    """Returns {"mission_sequence_order", "mission_cycle_id", "position",
    "total"} for `cell_id`, printing the selection as required."""
    common_missions = _find_common_missions(cell_id)

    if cell_id in MISSION_OVERRIDE:
        forced = int(MISSION_OVERRIDE[cell_id])
        per_framework_sets = _cell_mission_sets(cell_id)
        present_anywhere = any(forced in missions for missions in per_framework_sets.values())
        if not present_anywhere:
            raise ValueError(
                f"MISSION_OVERRIDE[{cell_id!r}] = {forced} but no loaded framework has this "
                f"mission_sequence_order for {cell_id}."
            )
        missing_from = [fw for fw, missions in per_framework_sets.items() if forced not in missions]
        if missing_from:
            warnings.warn(
                f"MISSION_OVERRIDE[{cell_id!r}] = {forced} is missing from {missing_from}; "
                f"those frameworks will be omitted for this cell's subplot."
            )
        selected = forced
        total = len(common_missions) if common_missions else len(per_framework_sets)
        position = (common_missions.index(selected) + 1) if selected in common_missions else None
    else:
        if not common_missions:
            raise ValueError(f"No common test mission found across loaded frameworks for {cell_id}.")
        if MISSION_SELECTION_MODE != "midpoint":
            raise ValueError(f"Unsupported MISSION_SELECTION_MODE: {MISSION_SELECTION_MODE!r}")
        idx = (len(common_missions) - 1) // 2
        selected = common_missions[idx]
        total = len(common_missions)
        position = idx + 1

    mission_cycle_id = None
    for frame in all_predictions.values():
        match = frame[
            (frame[COL_CELL_ID] == cell_id) & (frame[COL_MISSION_SEQUENCE_ORDER] == selected)
        ]
        if not match.empty:
            mission_cycle_id = match[COL_MISSION_ID].iloc[0]
            break

    position_str = f"{position} / {total}" if position is not None and total else "n/a"
    print(
        f"{cell_id}:\n"
        f"  selected mission = {mission_cycle_id}\n"
        f"  mission_sequence_order = {selected}\n"
        f"  position = {position_str}"
    )
    return {
        "mission_sequence_order": selected,
        "mission_cycle_id": mission_cycle_id,
        "position": position,
        "total": total,
    }


def _resolve_time_column(frame: pd.DataFrame) -> str | None:
    for candidate in COL_ELAPSED_TIME_CANDIDATES:
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None


def _mission_frame(frame: pd.DataFrame, cell_id: str, mission_order: int) -> pd.DataFrame:
    return frame[
        (frame[COL_CELL_ID] == cell_id) & (frame[COL_MISSION_SEQUENCE_ORDER] == mission_order)
    ]


def _mission_xy(frame: pd.DataFrame, column: str, time_col: str | None) -> tuple[np.ndarray, np.ndarray]:
    """Real elapsed seconds on x (never min-max normalized), the raw
    per-framework prediction points on y -- no interpolation, no
    point-count matching across frameworks."""
    if time_col is not None:
        ordered = frame.sort_values(time_col)
        x = ordered[time_col].to_numpy(dtype=float)
    else:
        ordered = frame.sort_values(COL_TIMESTEP_INDEX)
        x = np.arange(len(ordered), dtype=float)
    y = ordered[column].to_numpy(dtype=float)
    return x, y


In [ ]:
"""Code Cell 5 -- plotting functions."""


def _downsample(x: np.ndarray, y: np.ndarray, every_n: int) -> tuple[np.ndarray, np.ndarray]:
    """Display-only thinning: keeps every `every_n`-th point and always
    forces in the last point. Never mutates the underlying results."""
    if every_n <= 1 or len(x) <= 1:
        return x, y
    idx = np.arange(0, len(x), every_n)
    if idx[-1] != len(x) - 1:
        idx = np.append(idx, len(x) - 1)
    return x[idx], y[idx]


def _choose_reference_framework(mission_frames: dict[str, pd.DataFrame]) -> str | None:
    """Prefer an all_timesteps framework (the most complete sampling of the
    mission); among those (or, failing that, among whatever is available)
    prefer the one with the most points for this mission."""
    if not mission_frames:
        return None
    all_timestep_candidates = [
        fw for fw in mission_frames if EVALUATION_UNIT.get(fw) == "all_timesteps"
    ]
    pool = all_timestep_candidates if all_timestep_candidates else list(mission_frames)
    return max(pool, key=lambda fw: len(mission_frames[fw]))


def check_reference_soc_consistency(
    mission_frames: dict[str, pd.DataFrame], reference_framework: str | None, cell_id: str
) -> None:
    """Cross-framework sanity check for the single selected mission.
    Matches observations by source_row_right when both frameworks have it
    (the exact, canonical join key); otherwise falls back to a nearest-time
    match on the shared elapsed-time column. Never requires equal row
    counts between frameworks -- only compares the observations that can
    actually be matched."""
    if reference_framework is None or len(mission_frames) < 2:
        return
    ref_frame = mission_frames[reference_framework]
    for framework, frame in mission_frames.items():
        if framework == reference_framework:
            continue
        if COL_SOURCE_ROW_RIGHT in ref_frame.columns and COL_SOURCE_ROW_RIGHT in frame.columns:
            merged = ref_frame[[COL_SOURCE_ROW_RIGHT, COL_SOC_REFERENCE]].merge(
                frame[[COL_SOURCE_ROW_RIGHT, COL_SOC_REFERENCE]],
                on=COL_SOURCE_ROW_RIGHT,
                how="inner",
                suffixes=("_ref", "_other"),
            )
        else:
            time_col = _resolve_time_column(ref_frame)
            if time_col is None or time_col not in frame.columns:
                continue
            merged = pd.merge_asof(
                ref_frame[[time_col, COL_SOC_REFERENCE]].sort_values(time_col),
                frame[[time_col, COL_SOC_REFERENCE]].sort_values(time_col),
                on=time_col,
                direction="nearest",
                suffixes=("_ref", "_other"),
                tolerance=1.0,
            ).dropna()
        if merged.empty:
            continue
        mismatch = ~np.isclose(
            merged[f"{COL_SOC_REFERENCE}_ref"],
            merged[f"{COL_SOC_REFERENCE}_other"],
            atol=REFERENCE_SOC_MISMATCH_TOLERANCE,
            equal_nan=True,
        )
        if mismatch.any():
            warnings.warn(
                f"[{cell_id}] reference SOC mismatch between {reference_framework} and "
                f"{framework} on {int(mismatch.sum())}/{len(mismatch)} matched observations."
            )


def plot_single_mission_soc(
    ax: plt.Axes,
    cell_id: str,
    mission_info: dict,
    *,
    every_n: int = PLOT_EVERY_N,
) -> tuple[dict[str, pd.DataFrame], str | None]:
    """Draws Reference SOC once, plus one line per framework that has data
    for this cell's selected mission, onto `ax`. Only line plots -- no
    fill_between / area fills. Does not create or show the figure."""
    mission_order = mission_info["mission_sequence_order"]

    mission_frames: dict[str, pd.DataFrame] = {}
    for framework in FRAMEWORK_ORDER:
        frame = all_predictions.get(framework)
        if frame is None:
            continue
        sub = _mission_frame(frame, cell_id, mission_order)
        if sub.empty:
            warnings.warn(
                f"[{cell_id}] {framework} has no rows for mission_sequence_order="
                f"{mission_order}; skipping this framework for this cell."
            )
            continue
        assert sub[COL_MISSION_SEQUENCE_ORDER].nunique() == 1, (
            f"[{cell_id}/{framework}] expected exactly one mission in this subplot's data"
        )
        mission_frames[framework] = sub

    reference_framework = _choose_reference_framework(mission_frames)
    check_reference_soc_consistency(mission_frames, reference_framework, cell_id)

    x_label = "Elapsed time within mission (s)"
    time_col_for_label = None
    if reference_framework is not None:
        ref_frame = mission_frames[reference_framework]
        time_col_for_label = _resolve_time_column(ref_frame)
        x_ref, y_ref = _mission_xy(ref_frame, COL_SOC_REFERENCE, time_col_for_label)
        x_ref_d, y_ref_d = _downsample(x_ref, y_ref, every_n)
        ax.plot(
            x_ref_d, y_ref_d, color=REFERENCE_COLOR, linewidth=2.0, label="Reference SOC", zorder=5
        )

    for framework, sub in mission_frames.items():
        time_col = _resolve_time_column(sub)
        x, y = _mission_xy(sub, COL_SOC_PREDICTION, time_col)
        x_d, y_d = _downsample(x, y, every_n)
        ax.plot(
            x_d,
            y_d,
            color=FRAMEWORK_COLORS.get(framework),
            linewidth=1.4,
            alpha=0.9,
            label=framework,
        )
        if time_col is None:
            x_label = "Sample index within mission"

    ax.set_ylim(0, 100)
    ax.set_xlabel(x_label)
    ax.set_ylabel("SOC (%)")
    ax.set_title(cell_id)
    ax.annotate(
        f"Mission order: {mission_order}",
        xy=(0.98, 0.03),
        xycoords="axes fraction",
        ha="right",
        va="bottom",
        fontsize=7,
        color="#555555",
    )
    return mission_frames, reference_framework


def plot_all_cells_soc(mission_info_by_cell: dict[str, dict]) -> plt.Figure:
    """Combined 7-cell figure: a 3x6 GridSpec, first two rows split into
    three 2-column-wide subplots each, third row's single subplot centered
    in columns [2:4] (same width as the others, not bottom-left). Each
    subplot is one cell's single representative mission."""
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(3, 6, figure=fig, hspace=0.4, wspace=0.3)
    slot_specs = [
        gs[0, 0:2], gs[0, 2:4], gs[0, 4:6],
        gs[1, 0:2], gs[1, 2:4], gs[1, 4:6],
        gs[2, 2:4],
    ]

    axes = []
    for cell_id, spec in zip(CELL_IDS, slot_specs):
        ax = fig.add_subplot(spec)
        plot_single_mission_soc(ax, cell_id, mission_info_by_cell[cell_id])
        axes.append(ax)

    handles_by_label: dict[str, object] = {}
    legend_order = ["Reference SOC", *FRAMEWORK_ORDER]
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            handles_by_label.setdefault(label, handle)
    ordered_labels = [label for label in legend_order if label in handles_by_label]
    ordered_handles = [handles_by_label[label] for label in ordered_labels]

    fig.legend(
        ordered_handles,
        ordered_labels,
        loc="lower center",
        ncol=min(len(ordered_labels), 4),
        bbox_to_anchor=(0.5, -0.02),
        frameon=False,
    )
    fig.suptitle(
        "SOC Trajectories: Panasonic25 -> CMU eVTOL Transfer (one representative mission per cell)",
        y=0.99,
    )
    return fig


In [ ]:
"""Diagnostics summary -- run before plotting.

Prints one row per cell: which mission was selected, how many frameworks
agreed on a common mission set, and how many prediction points each loaded
framework actually contributes for that mission (their evaluation-unit
conventions differ on purpose -- see EVALUATION_UNIT -- so these counts are
expected to differ across frameworks, e.g. DARM's last_step_per_window vs.
SFTNN/Backbone's all_timesteps)."""

_diagnostic_rows = []
for _cell_id in CELL_IDS:
    _mission_info = select_representative_mission(_cell_id)
    _mission_frames = {
        framework: _mission_frame(frame, _cell_id, _mission_info["mission_sequence_order"])
        for framework, frame in all_predictions.items()
    }
    _mission_frames = {fw: sub for fw, sub in _mission_frames.items() if not sub.empty}
    _reference_framework = _choose_reference_framework(_mission_frames)

    for _sub in _mission_frames.values():
        assert _sub[COL_MISSION_SEQUENCE_ORDER].nunique() == 1

    _row = {
        "cell": _cell_id,
        "mission_order": _mission_info["mission_sequence_order"],
        "mission_cycle_id": _mission_info["mission_cycle_id"],
        "common_framework_count": len(_mission_frames),
        "reference_source": _reference_framework,
        "n_reference_points": len(_mission_frames.get(_reference_framework, [])) if _reference_framework else 0,
    }
    for _framework in FRAMEWORK_ORDER:
        _row[f"{_framework}_points"] = len(_mission_frames.get(_framework, []))
    _diagnostic_rows.append(_row)

diagnostic_summary = pd.DataFrame(_diagnostic_rows)
print()
print(diagnostic_summary.to_string(index=False))


## Per-cell trajectories

Seven independent cells below (not a loop), one per target cell in
`CELL_IDS`, so each can be re-run/edited on its own. Each subplot shows
exactly one representative test mission (see `select_representative_mission`
above) -- not the whole test-set lifespan.

In [ ]:
"""Code Cell 6 -- CELL_IDS[0]."""

_cell_id = CELL_IDS[0]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 7 -- CELL_IDS[1]."""

_cell_id = CELL_IDS[1]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 8 -- CELL_IDS[2]."""

_cell_id = CELL_IDS[2]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 9 -- CELL_IDS[3]."""

_cell_id = CELL_IDS[3]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 10 -- CELL_IDS[4]."""

_cell_id = CELL_IDS[4]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 11 -- CELL_IDS[5]."""

_cell_id = CELL_IDS[5]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


In [ ]:
"""Code Cell 12 -- CELL_IDS[6]."""

_cell_id = CELL_IDS[6]
selected_mission = select_representative_mission(_cell_id)

fig, ax = plt.subplots(figsize=(9, 4.5))
_mission_frames, _reference_framework = plot_single_mission_soc(ax, _cell_id, selected_mission)
assert len({sub[COL_MISSION_SEQUENCE_ORDER].iat[0] for sub in _mission_frames.values()}) <= 1
ax.legend(loc="upper right", frameon=False, fontsize=8)
plt.tight_layout()
if SAVE_INDIVIDUAL_CELL_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"soc_trajectory_{_cell_id}.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## Combined figure (all seven cells)

In [ ]:
"""Code Cell 13 -- combined 3+3+1 figure across all seven cells."""

_mission_info_by_cell = {cell_id: select_representative_mission(cell_id) for cell_id in CELL_IDS}
combined_fig = plot_all_cells_soc(_mission_info_by_cell)
plt.show()


## Export

In [ ]:
"""Code Cell 14 -- export the combined figure to PNG and PDF."""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / "soc_trajectory_all_cells.png"
pdf_path = OUTPUT_DIR / "soc_trajectory_all_cells.pdf"
combined_fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
combined_fig.savefig(pdf_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {png_path}")
print(f"Saved: {pdf_path}")
